In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
from IPython.display import display

from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
sns.set_palette("muted")
PALETTE = sns.color_palette("muted", 9)
ALL_FIGS = []
print("Bibliotheken geladen")


# ── ZELLE 3: Einlesen & Region-Spalte hinzufügen ────────────
DATEIEN = {
    "Barcelona":  "data/raw/listings_barcelona.csv",
    "Euskadi":    "data/raw/listings_euskadi.csv",
    "Girona":     "data/raw/listings_girona.csv",
    "Madrid":     "data/raw/listings_madrid.csv",
    "Malaga":     "data/raw/listings_malaga.csv",
    "Mallorca":   "data/raw/listings_mallorca.csv",
    "Menorca":    "data/raw/listings_menorca.csv",
    "Sevilla":    "data/raw/listings_sevilla.csv",
    "Valencia":   "data/raw/listings_valencia.csv",
}

frames = []
print(f"{'Region':<12} {'Zeilen':>8}  {'Spalten':>8}  {'Price-NaN':>10}  {'Price>1000':>11}")
print("─" * 58)

for region, datei in DATEIEN.items():
    df_tmp = pd.read_csv(datei)
    df_tmp.insert(0, "region", region)
    n_nan    = df_tmp["price"].isna().sum()
    n_out    = (df_tmp["price"] > 1000).sum()
    print(f"{region:<12} {len(df_tmp):>8,}  {df_tmp.shape[1]:>8}  {n_nan:>10,}  {n_out:>11,}")
    frames.append(df_tmp)

df_raw = pd.concat(frames, ignore_index=True)
print("─" * 58)
print(f"{'GESAMT':<12} {len(df_raw):>8,}")
print(f"\n Rohdatensatz Shape: {df_raw.shape}")
display(df_raw.head())


# ════════════════════════════════════════════════════════════
#  TEIL 1 – EDA PRO REGION (ROH)
# ════════════════════════════════════════════════════════════

# ── ZELLE 4: Listings pro Region ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_raw["region"].value_counts().sort_values()
counts.plot.barh(ax=axes[0], color=PALETTE)
axes[0].set_title("Anzahl Listings je Region")
axes[0].set_xlabel("Anzahl")

counts.plot.pie(
    autopct="%1.1f%%", ax=axes[1],
    colors=PALETTE, ylabel="", startangle=90
)
axes[1].set_title("Anteil je Region")
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)


# ── ZELLE 5: Fehlende Werte gesamt & je Region ──────────────
missing = (
    df_raw.drop(columns="region")
    .isnull().sum()
    .rename("fehlend").to_frame()
)
missing["anteil_%"] = (missing["fehlend"] / len(df_raw) * 100).round(2)
missing = missing[missing["fehlend"] > 0].sort_values("fehlend", ascending=False)
print("FEHLENDE WERTE (Gesamtdatensatz)")
display(missing)

# Fehlende price-Werte je Region
price_nan_region = (
    df_raw.groupby("region")["price"]
    .apply(lambda x: x.isna().sum())
    .rename("price_NaN")
    .reset_index()
)
print("\nFehlende Price-Werte je Region:")
display(price_nan_region)

# Missingno Heatmap
fig, ax = plt.subplots(figsize=(15, 5))
msno.bar(df_raw.drop(columns="region"), ax=ax, color="#5B9BD5", fontsize=9)
plt.title("Vollständigkeit je Spalte (Gesamtdatensatz)", fontsize=13, pad=12)
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)


# ── ZELLE 6: Preisverteilung je Region (Roh) ────────────────
# Hinweis: Die x-Achse wird bewusst auf 0-600€ begrenzt, da der
# überwiegende Teil aller Listings in diesem Bereich liegt.
# Ohne diese Begrenzung würde die x-Achse durch einzelne
# Extremwerte (bis zu 90.000€) so stark gestreckt, dass die
# eigentliche Verteilung nicht mehr erkennbar wäre.
PRICE_LIMIT = 600
 
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
 
for i, (region, color) in enumerate(zip(DATEIEN.keys(), PALETTE)):
    sub = df_raw[df_raw["region"] == region]["price"].dropna()
    n_above = (sub > PRICE_LIMIT).sum()
    axes[i].hist(sub, bins=60, range=(0, PRICE_LIMIT), color=color, edgecolor="white")
    axes[i].set_xlim(0, PRICE_LIMIT)
    axes[i].set_title(f"{region}  (n={len(sub):,}, {n_above:,} > {PRICE_LIMIT}€ n. dargestellt)")
    axes[i].set_xlabel("Preis (€)")
    axes[i].set_ylabel("Anzahl")
 
plt.suptitle(f"Preisverteilung je Region – ROH (x-Achse begrenzt auf 0–{PRICE_LIMIT}€)", fontsize=14, y=1.02)
fig.text(
    0.5, 0.995,
    "n = Anzahl Listings mit vorhandenem Preiseintrag (ohne fehlende Werte). "
    "Werte oberhalb der x-Achse sind nicht abgebildet, aber in n enthalten.",
    ha="center", fontsize=9, color="gray"
)
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)

# ── ZELLE 7: Boxplots Preis je Region (Roh) – gesamt & je Zimmertyp ─
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
 
# Links: Gesamtüberblick je Region
order_region = (
    df_raw.groupby("region")["price"]
    .median().sort_values(ascending=False).index
)
sns.boxplot(
    data=df_raw[df_raw["price"].notna()],
    x="region", y="price", order=order_region,
    ax=axes[0], showfliers=False
)
axes[0].set_title("Preis je Region (ohne Extreme, Whisker = IQR×1.5)")
axes[0].set_xlabel("")
axes[0].set_ylabel("Preis (€)")
axes[0].tick_params(axis="x", rotation=30)
 
# Rechts: zusätzlich nach Zimmertyp aufgeschlüsselt
sns.boxplot(
    data=df_raw[df_raw["price"].notna()],
    x="region", y="price", hue="room_type", order=order_region,
    ax=axes[1], showfliers=False
)
axes[1].set_title("Preis je Region & Zimmertyp (ohne Extreme)")
axes[1].set_xlabel("")
axes[1].set_ylabel("Preis (€)")
axes[1].set_ylim(0, 800)
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="Zimmertyp", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
axes[1].text(
    0.98, 0.97,
    "Hinweis: Menorca / Hotel room (n=9, Q3≈9.115€) liegt\n"
    "außerhalb des dargestellten Bereichs (sehr kleine\n"
    "Stichprobe, IQR statistisch nicht belastbar) und wird\n"
    "hier abgeschnitten.",
    transform=axes[1].transAxes, ha="right", va="top", fontsize=8,
    style="italic", bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)
 
fig.suptitle(
    "Preisboxplots je Region – ROH\n"
    "Hinweis: 'Entire home/apt' ist systematisch teurer als 'Private/Shared room' – "
    "dies wird beim Ausreißerfilter (Zelle 12) berücksichtigt.",
    fontsize=12, y=1.04
)
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)


# ── ZELLE 9: Zimmertypen je Region ──────────────────────────
room_region = (
    df_raw.groupby(["region", "room_type"])
    .size()
    .reset_index(name="n")
)
pivot = room_region.pivot(index="region", columns="room_type", values="n").fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 6))
pivot_pct.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white")
ax.set_title("Zimmertyp-Anteile je Region (%)")
ax.set_ylabel("Anteil (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Zimmertyp", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)


# ── ZELLE 10: Geografische Übersicht (mit Küstenlinien) ─────
import cartopy.crs as ccrs

fig, ax = plt.subplots(
    figsize=(12, 8),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

ax.coastlines(resolution='10m', linewidth=1)

for (region, grp), color in zip(df_raw.groupby("region"), PALETTE):
    ax.scatter(
        grp["longitude"],
        grp["latitude"],
        color=color,
        alpha=0.4,
        s=5,
        label=region,
        transform=ccrs.PlateCarree()
    )

ax.set_extent([-10, 5, 35, 45], crs=ccrs.PlateCarree())
ax.set_title("Geografische Verteilung aller Listings in Spanien", fontsize=14)
ax.legend(markerscale=3, loc="lower left")
ALL_FIGS.append(fig)
plt.close(fig)


# ── ZELLE 11: Korrelationsmatrix (Gesamtdatensatz roh) ──────
numeric_cols = df_raw.select_dtypes(include="number").columns.tolist()
corr = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, ax=ax, linewidths=0.5
)
ax.set_title("Korrelationsmatrix – numerische Merkmale (Gesamtdatensatz)")
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)


# ════════════════════════════════════════════════════════════
#  TEIL 2 – DATENBEREINIGUNG
# ════════════════════════════════════════════════════════════

# ── ZELLE 12: Bereinigung ───────────────────────────────────
df = df_raw.copy()

print("=" * 65)
print("DATENBEREINIGUNG – PROTOKOLL")
print("=" * 65)

n_start = len(df)
print(f"\n[START]  Zeilen gesamt:               {n_start:>8,}")

# --- Schritt 1: Zeilen ohne Preis entfernen ---
n_no_price = df["price"].isna().sum()
df = df[df["price"].notna()].copy()
print(f"\n[1] Entfernt – kein Preis (NaN):      {n_no_price:>8,}")
print(f"    Verbleibend:                       {len(df):>8,}")

# --- Übersicht je Region ---
print(f"\n{'─'*65}")
print(f"VERBLEIBENDE ZEILEN JE REGION (bereinigt)")
region_counts = df["region"].value_counts().sort_values(ascending=False)
for region, n in region_counts.items():
    print(f"    {region:<14} {n:>8,}")
print(f"{'─'*65}")
print(f"    {'GESAMT':<14} {len(df):>8,}   (Reduktion: {(n_start-len(df))/n_start*100:.1f} %)")


# ── ZELLE 13: Boxplot Preis je Region – Roh vs. Bereinigt ───
fig, ax = plt.subplots(figsize=(15, 6))

df_compare = pd.concat([
    df_raw[df_raw["price"].notna()].assign(status="Roh"),
    df.assign(status="Bereinigt")
])

order_clean = df.groupby("region")["price"].median().sort_values(ascending=False).index

sns.boxplot(
    data=df_compare,
    x="region", y="price",
    hue="status", order=order_clean,
    palette={"Roh": "#FF6B6B", "Bereinigt": "#5B9BD5"},
    showfliers=False, ax=ax
)
ax.set_title("Preis je Region: Roh vs. Bereinigt (Whisker = IQR×1.5, ohne Extremwerte)")
ax.set_ylabel("Preis (€)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Datensatz")
plt.tight_layout()
ALL_FIGS.append(fig)
plt.close(fig)

print("\nPreis-Statistik je Region (bereinigt):")
display(
    df.groupby("region")["price"]
    .describe()
    .round(2)
    .sort_values("50%", ascending=False)
)


# ── ZELLE 14: Export ────────────────────────────────────────
output_file = "listings_spanien_cleaned.csv"
df.to_csv(output_file, index=False)
print(f" Bereinigte Datei gespeichert: {output_file}")
print(f"   Zeilen: {len(df):,}  |  Spalten: {df.shape[1]}  |  Regionen: {df['region'].nunique()}")


# ── ZELLE 15: Alle Plots in eine PDF exportieren ────────────
pdf_file = "eda_airbnb_spanien_plots.pdf"

with PdfPages(pdf_file) as pdf:
    for fig in ALL_FIGS:
        pdf.savefig(fig, bbox_inches="tight")

print(f" Alle Grafiken wurden in einer PDF gespeichert: {pdf_file}")
print(f" Enthaltene Seiten/Plots: {len(ALL_FIGS)}")


✅ Bibliotheken geladen
Region         Zeilen   Spalten   Price-NaN   Price>1000
──────────────────────────────────────────────────────────
Barcelona      19,410        19       4,134          174
Euskadi         6,334        19         683           60
Girona         21,060        19       3,242          906
Madrid         25,000        19       6,047           88
Malaga          9,714        19         899          228
Mallorca       15,034        19           7        2,483
Menorca         3,679        19         447          213
Sevilla         8,215        19         634           87
Valencia        7,844        19         865           63
──────────────────────────────────────────────────────────
GESAMT        116,290

✅ Rohdatensatz Shape: (116290, 19)


,region,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,Barcelona,18674,Huge flat for 8 people close to Sagrada Familia,71615,Mireia,Eixample,la Sagrada Família,41.405560,2.17262,Entire home/apt,210.0,1,51,2025-07-31,0.34,26,80,7,ESFCTU000008058000039706000000000000000HUTB-00...
1,Barcelona,23197,"Forum CCIB DeLuxe, Spacious, Large Balcony, relax",90417,Etain (Marnie),Sant Martí,el Besòs i el Maresme,41.412432,2.21975,Entire home/apt,285.0,3,91,2025-09-08,0.52,1,289,12,ESFCTU000008106000547162000000000000000000HUTB...
2,Barcelona,32711,Sagrada Familia area - Còrsega 1,135703,Nick,Gràcia,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,Entire home/apt,170.0,1,152,2025-08-08,0.88,2,64,23,HUTB-001722
3,Barcelona,34241,Stylish Top Floor Apartment - Ramblas Plaza Real,73163,Andres,Ciutat Vella,el Barri Gòtic,41.380620,2.17517,Entire home/apt,110.0,31,25,2024-11-05,0.14,3,333,5,Exempt
4,Barcelona,34981,VIDRE HOME PLAZA REAL on LAS RAMBLAS,73163,Andres,Ciutat Vella,el Barri Gòtic,41.379780,2.17623,Entire home/apt,333.0,5,271,2025-08-19,1.49,3,335,23,ESFCTU000008119000093652000000000000000HUTB-00...


FEHLENDE WERTE (Gesamtdatensatz)


,fehlend,anteil_%
neighbourhood_group,49487,42.55
license,29735,25.57
reviews_per_month,20187,17.36
last_review,20187,17.36
price,16958,14.58
host_name,301,0.26



Fehlende Price-Werte je Region:


...

DATENBEREINIGUNG – PROTOKOLL

[START]  Zeilen gesamt:                116,290

[1] Entfernt – kein Preis (NaN):        16,958
    Verbleibend:                         99,332

[2] Entfernt – IQR-Ausreißer (Faktor=3, je Region+Zimmertyp):    6,403
    Verbleibend:                         92,929

[3] Entfernt – Duplikate:                    0
    Verbleibend:                         92,929

─────────────────────────────────────────────────────────────────
VERBLEIBENDE ZEILEN JE REGION (bereinigt)
    Madrid           18,219
    Girona           16,424
    Barcelona        14,694
    Mallorca         12,847
    Malaga            8,298
    Sevilla           7,304
    Valencia          6,698
    Euskadi           5,434
    Menorca           3,011
─────────────────────────────────────────────────────────────────
    GESAMT           92,929   (Reduktion: 20.1 %)

Preis-Statistik je Region (bereinigt):


,count,mean,std,min,25%,50%,75%,max
region,,,,,,,,
Mallorca,12847.0,306.44,232.05,10.0,156.0,238.0,376.5,1680.0
Menorca,3011.0,218.93,331.17,23.0,98.0,148.0,259.0,9243.0
Euskadi,5434.0,150.33,93.46,14.0,87.0,126.0,186.0,570.0
Barcelona,14694.0,149.58,106.25,9.0,68.0,125.0,203.0,665.0
Sevilla,7304.0,138.07,76.26,12.0,89.0,122.0,172.0,1000.0
Girona,16424.0,152.96,102.29,10.0,86.0,120.0,183.0,602.0
Madrid,18219.0,121.27,74.24,8.0,68.0,107.0,154.5,444.0
Valencia,6698.0,106.50,55.83,8.0,69.0,100.0,134.0,330.0
Malaga,8298.0,111.69,58.13,16.0,75.0,99.0,135.0,1000.0


✅ Bereinigte Datei gespeichert: listings_spanien_cleaned.csv
   Zeilen: 92,929  |  Spalten: 19  |  Regionen: 9


/usr/local/python/3.12.1/lib/python3.12/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ Alle Grafiken wurden in einer PDF gespeichert: eda_airbnb_spanien_plots.pdf
   Enthaltene Seiten/Plots: 10
